# Hybrid Search
Hybrid search combines traditional keyword-based search with semantic search to provide more accurate and relevant results. In the RAG application, it facilitates the discovery of relevant research articles based on user queries by integrating keyword-based search with semantic search capabilities. This integration enables the application to retrieve articles that match both keywords and semantic meaning, making it particularly useful for handling complex queries involving nuanced concepts, synonyms, and related ideas.

In this notebook, we will delve into the implementation details of the hybrid search approach in the RAG application, exploring how it leverages both keyword-based and semantic search techniques to provide a more effective search experience.

Here are the steps:

Loading chunked dataset
Sparse Index
Dense Index
Merging Results
Generating a reply with merged results

#### Visual improvements
We will use rich library to make the output more readable, and supress warning messages.

In [1]:
from rich.console import Console
from rich_theme_manager import Theme, ThemeManager
import pathlib

theme_dir = pathlib.Path("themes")
theme_manager = ThemeManager(theme_dir=theme_dir)
dark = theme_manager.get("dark")

# Create a console with the dark theme
console = Console(theme=dark)
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

### Hybrid Search - Sparse Index
We will use bm25 supported database to complement the semantic search with the vector database.

In [3]:
import bm25s
from bm25s.tokenization import Tokenizer, Tokenized
import Stemmer  # optional: for stemming

##### Loading the chunks from the previous steps
We will use the chunks from the AI Arxiv dataset, we used before. These chunks were split using semantic chunking and enriched with context.

In [4]:
import json
corpus_json = json.load(open('data/corpus.json'))

##### Creating the Sparse Index
We will use an in-memory index using BM25. Many (vector) databases support BM25 natively, and many others support indexing and searching on calculated sparse vectors.

In this example, we will also define a stemmer and stop-words to clean up the text and better select the tokens/terms that will be indexed in the sparse index.

In [5]:
corpus_text = [doc["text"] for doc in corpus_json]

# optional: create a stemmer
english_stemmer = Stemmer.Stemmer("english")

# Initialize the Tokenizer with the stemmer
sparse_tokenizer = Tokenizer(
    stemmer=english_stemmer,
    lower=True, # lowercase the tokens
    stopwords="english",  # or pass a list of stopwords
    splitter=r"\w+",  # by default r"(?u)\b\w\w+\b", can also be a function
)
console.print(sparse_tokenizer.stopwords)

(
    'a',
    'an',
    'and',
    'are',
    'as',
    'at',
    'be',
    'but',
    'by',
    'for',
    'if',
    'in',
    'into',
    'is',
    'it',
    'no',
    'not',
    'of',
    'on',
    'or',
    'such',
    'that',
    'the',
    'their',
    'then',
    'there',
    'these',
    'they',
    'this',
    'to',
    'was',
    'will',
    'with'
)

In [6]:
# Tokenize the corpus and only keep the ids (faster and saves memory)
corpus_sparse_tokens = (
    sparse_tokenizer
    .tokenize(
        corpus_text, 
        update_vocab=True, # update the vocab as we tokenize
        return_as="ids"
    )
)

# Create the BM25 retriever and attach your corpus_json to it
sparse_index = bm25s.BM25(corpus=corpus_json)
# Now, index the corpus_tokens (the corpus_json is not used yet)
sparse_index.index(corpus_sparse_tokens)

Tokenize texts:   0%|          | 0/49 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/49 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/49 [00:00<?, ?it/s]

In [7]:
vocab_dict = sparse_tokenizer.get_vocab_dict()
console.print(f"The tokenizer vocabulary includes {len(vocab_dict)} tokens/terms")

focus_token = 'context'
focus_token_index = vocab_dict.get(focus_token)
console.print(f"The index of the {focus_token} is {focus_token_index}")

The tokenizer vocabulary includes 1715 tokens/terms

The index of the context is 128

The tokenizer can encode (convert the text into ids) and decode (convert the ids back into text).

In [8]:
console.print(sparse_tokenizer.decode([[focus_token_index]]))

[['context']]

#### Exploring the Sparse Index

In [9]:
console.print(sparse_index.scores)

{
    'data': array([0.7581095, 1.0530901, 1.153632 , ..., 1.3138028, 1.3138028,
       1.3138028], dtype=float32),
    'indices': array([ 0, 10, 12, ..., 47, 47, 47], dtype=int32),
    'indptr': array([   0,    0,   12, ..., 4533, 4534, 4535]),
    'num_docs': 49
}

For each token, the index holds the list of documents (chunks) that include it, and the score of that token in that document (chunk).

In [10]:
from rich.table import Table
from rich.style import Style

token_index = vocab_dict.get(focus_token)
console.print(f"Index of the token `{focus_token}` in the BM25 retriever: {token_index}")
score_index = sparse_index.scores.get('indptr')[token_index]
next_score_index = sparse_index.scores.get('indptr')[token_index+1]

table = Table(title=f"Document Scores for `{focus_token}`")

table.add_column("Document ID", justify="right", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="bright_green")

max_score = max(sparse_index.scores['data'][score_index:next_score_index])
# Define styles for specific rows
highlight_style = Style(bgcolor="yellow")

for i in range(score_index, next_score_index):
    doc_id = sparse_index.scores['indices'][i]
    doc_score = sparse_index.scores['data'][i]
    if doc_score == max_score:
        table.add_row(
            str(doc_id),
            str(doc_score), style=highlight_style
        )
    else:
        table.add_row(
            str(doc_id),
            str(doc_score)
        )

console.print(table)

Index of the token `context` in the BM25 retriever: 128

    Document Scores for     
         `context`          
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Document ID ┃      Score ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│           0 │ 0.37561262 │
│           2 │  0.6274637 │
│           3 │ 0.47197592 │
│           4 │  0.7962141 │
│          17 │ 0.42602202 │
│          18 │  0.9924868 │
│          19 │  0.9709488 │
│          22 │ 0.42602202 │
│          30 │  0.5823246 │
│          36 │ 0.63236547 │
│          40 │  0.6132039 │
│          42 │ 0.55822957 │
│          43 │  0.4534678 │
└─────────────┴────────────┘

#### Searching the Sparse Index
As we are doing in the dense index, we need to tokenize and encode the query text:

In [11]:
# Query the corpus
query = "What is context size of Mixtral?"
query_tokens = (
    sparse_tokenizer
    .tokenize(
        [query], 
        update_vocab=False, 
        return_as="ids"
    )
)

console.print(query_tokens)

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

[[128, 129, 16]]

And use the encoded query to search the sparse index:

In [12]:
# Query the corpus
sparse_results, sparse_scores = sparse_index.retrieve(query_tokens, k=10)

for i in range(sparse_results.shape[1]):
    doc, score = sparse_results[0, i], sparse_scores[0, i]
    console.print(f"Rank {i+1} (score: {score:.2f}): {doc}")

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Rank 1 (score: 1.90): {'id': 18, 'text': "Table 4: Comparison of Mixtral with Llama on Multilingual Benchmarks. On 
ARC Challenge, Hellaswag, and MMLU, Mixtral outperforms Llama 2 70B on 4 languages: French, German, Spanish, and 
Italian. # 3.2 Long range performance To assess the capabilities of Mixtral to tackle long context, we evaluate it 
on the passkey retrieval task introduced in [23], a synthetic task designed to measure the ability of the model to 
retrieve a passkey inserted randomly in a long prompt. Results in Figure 4 (Left) show that Mixtral achieves a 100%
retrieval accuracy regardless of the context length or the position of passkey in the sequence. Figure 4 (Right) 
shows that the perplexity of Mixtral on a subset of the proof-pile dataset [2] decreases monotonically as the size 
of the context increases. Passkey Performance ry 0.8 0.6 04 0.2 0.0 OK 4K 8K 12K 16K 20K 24K 28K Seq Len Passkey 
Loc\n\nThis chunk is part of the results section of the document, specifically focusing on the performance of the 
Mixtral model in multilingual benchmarks and its capabilities in handling long context tasks, including the passkey
retrieval task. It highlights Mixtral's superior performance compared to Llama 2 70B in various languages and 
presents quantitative results related to its accuracy and perplexity in long context scenarios.", 'metadata': 
{'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

Rank 2 (score: 1.73): {'id': 2, 'text': 'expertsâ ) to process the token and combine their output additively. This 
technique increases the number of parameters of a model while controlling cost and latency, as the model only uses 
a fraction of the total set of parameters per token. Mixtral is pretrained with multilingual data using a context 
size of 32k tokens. It either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several 
benchmarks. In particular, Mixture of Experts Layer i gating inputs af outputs router expert\n\nThis chunk is part 
of the section discussing the architecture and functionality of the Mixtral 8x7B model, specifically focusing on 
the Mixture of Experts (MoE) mechanism, which allows the model to utilize a subset of its parameters for each token
processed, enhancing efficiency and performance across various benchmarks.', 'metadata': {'title': 'Mixtral of 
Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

Rank 3 (score: 1.31): {'id': 1, 'text': "chat model on human bench- marks. Both the base and instruct models are 
released under the Apache 2.0 license. Code: https://github.com/mistralai/mistral-src Webpage: 
https://mistral.ai/news/mixtral-of-experts/ # Introduction In this paper, we present Mixtral 8x7B, a sparse mixture
of experts model (SMoE) with open weights, licensed under Apache 2.0. Mixtral outperforms Llama 2 70B and GPT-3.5 
on most benchmarks. As it only uses a subset of its parameters for every token, Mixtral allows faster inference 
speed at low batch-sizes, and higher throughput at large batch-sizes. Mixtral is a sparse mixture-of-experts 
network. It is a decoder-only model where the feedforward block picks from a set of 8 distinct groups of 
parameters. At every layer, for every token, a router network chooses two of these groups (the â\n\nThe chunk is 
part of the introduction section of the document, which presents Mixtral 8x7B, a Sparse Mixture of Experts (SMoE) 
language model. It highlights the model's architecture, performance advantages over Llama 2 70B and GPT-3.5, and 
its efficient use of parameters for improved inference speed and throughput.", 'metadata': {'title': 'Mixtral of 
Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

Rank 4 (score: 1.10): {'id': 15, 'text': 'Size and Efficiency. We compare our performance to the Llama 2 family, 
aiming to understand Mixtral modelsâ efficiency in the cost-performance spectrum (see Figure 3). As a sparse 
Mixture- of-Experts model, Mixtral only uses 13B active parameters for each token. With 5x lower active parameters,
Mixtral is able to outperform Llama 2 70B across most categories. Note that this analysis focuses on the active 
parameter count (see Section 2.1), which is directly proportional to the inference compute cost, but does not 
consider the memory costs and hardware utilization. The memory costs for serving Mixtral are proportional to its 
sparse parameter count, 47B, which is still smaller than Llama 2 70B. As for device utilization, we note that the 
SMoEs layer introduces additional overhead due to the routing mechanism and due to the increased memory loads when 
running more than one expert per device. They are more suitable for batched workloads where one can reach a good 
degree of arithmetic intensity. Comparison with Llama 2 70B and GPT-3.5. In Table 3, we report the performance of 
Mixtral 8x7B compared to Llama 2 70B and GPT-3.5. We observe that Mixtral performs similarly or above the two other
models. On MMLU, Mixtral obtains a better performance, despite its significantly smaller capacity (47B tokens 
compared to 70B). For MT Bench, we report the performance of the latest GPT-3.5-Turbo model available, 
gpt-3.5-turbo-1106. 2Since Llama 2 34B was not open-sourced, we report results for Llama 1 34B.\n\nThis chunk is 
part of the "Results" section of the document, specifically discussing the size and efficiency of the Mixtral 8x7B 
model in comparison to the Llama 2 family and GPT-3.5. It highlights the active parameter count, performance 
metrics, and implications for memory costs and device utilization, emphasizing Mixtral\'s advantages in efficiency 
and performance across various benchmarks.', 'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 
'references': ['1905.07830']}}

...

Rank 6 (score: 1.02): {'id': 19, 'text': '3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 
30k Context length Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 
4K 8K 12K 16K 20K 24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance of 
Mixtral. (Left) Mixtral has 100% retrieval accuracy of the Passkey task regardless of the location of the passkey 
and length of the input sequence. (Right) The perplexity of Mixtral on the proof-pile dataset decreases 
monotonically as the context length increases.\n\nThis chunk is part of the "Results" section of the document, 
specifically discussing the long-range performance of the Mixtral 8x7B model on a passkey retrieval task and its 
perplexity on the proof-pile dataset, highlighting its effectiveness in handling long contexts.', 'metadata': 
{'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

Rank 7 (score: 0.85): {'id': 4, 'text': 'Instruct under the Apache 2.0 license1, free for academic and commercial 
usage, ensuring broad accessibility and potential for diverse applications. To enable the community to run Mixtral 
with a fully open-source stack, we submitted changes to the vLLM project, which integrates Megablocks CUDA kernels 
for efficient inference. Skypilot also allows the deployment of vLLM endpoints on any instance in the cloud. # 2 
Architectural details Mixtral is based on a transformer architecture [31] and uses the same modifications as 
described in [18], with the notable exceptions that Mix- tral supports a fully dense context length of 32k tokens, 
and the feed- forward blocks are replaced by Mixture-of-Expert layers (Section 2.1). The model architecture 
parameters are summarized in Table 1.\n\nThis chunk is part of the "Introduction" and "Architectural details" 
sections of the document, which discusses the Mixtral 8x7B model, its licensing, accessibility, and the 
architectural modifications made to the transformer framework, including the implementation of Mixture-of-Expert 
layers and context length specifications.', 'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 
'references': ['1905.07830']}}

Rank 8 (score: 0.67): {'id': 36, 'text': 'Quac: Question answering in context. arXiv preprint arXiv:1808.07036, 
2018. [6] Aidan Clark, Diego De Las Casas, Aurelia Guy, Arthur Mensch, Michela Paganini, Jordan Hoffmann, Bogdan 
Damoc, Blake Hechtman, Trevor Cai, Sebastian Borgeaud, et al. Unified scaling laws for routed language models. In 
International Conference on Machine Learning, pages 4057â 4086. PMLR, 2022. [7] Christopher Clark, Kenton Lee, 
Ming-Wei Chang, Tom Kwiatkowski, Michael Collins, and Kristina Toutanova.\n\nThe chunk is part of the references 
section of the document, which lists various academic papers and preprints relevant to the research on language 
models, including the Mixtral 8x7B model discussed throughout the document.', 'metadata': {'title': 'Mixtral of 
Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

Rank 9 (score: 0.61): {'id': 30, 'text': "= nri.|funct ional softinax'( weights, din=1, dtype=torch. float, 
).type_as|(inputs) results| = torch. zeros_ ike! linputs_squashe for i, expert in enunerate(self. experts): 
batch_idx,! nth_expert = torch. wnere( results  += weights [batch_i input s_squashed  ) return resutts:.view 
las{(inputs) class NoeLayer (nn. Module) = def _ init__(self, experts! List'{nri.Modulelly Super (Tz init_t assert 
len (experts) > 9) self.experts = nn. ModuleList((experits)) def forward(self, inputs: torch. Tensor)?\n\nThis 
chunk is part of the implementation details for the Mixture of Experts (MoE) layer in the Mixtral 8x7B model, 
specifically focusing on the forward method that processes inputs through the selected experts. It discusses the 
computation of weights and the aggregation of results from the experts, highlighting the model's architecture and 
functionality within the broader context of the paper's architectural details section.", 'metadata': {'title': 
'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

Rank 10 (score: 0.61): {'id': 40, 'text': 'Triviaqa: A large scale distantly supervised challenge dataset for 
reading comprehension. arXiv preprint arXiv:1705.03551, 2017. [20] Tom Kwiatkowski, Jennimaria Palomaki, Olivia 
Redfield, Michael Collins, Ankur Parikh, Chris Alberti, Danielle Epstein, Illia Polosukhin, Jacob Devlin, Kenton 
Lee, et al. Natural questions: a benchmark for question answering research. Transactions of the Association for 
Computational Linguistics, pages 453â 466, 2019. [21] Dmitry Lepikhin, HyoukJoong Lee, Yuanzhong Xu, Dehao Chen, 
Orhan Firat, Yanping Huang, Maxim Krikun, Noam Shazeer, and Zhifeng Chen.\n\nThe chunk is part of the references 
section of the document, which lists various academic papers and datasets relevant to the evaluation and 
benchmarking of language models, specifically in the context of reading comprehension and question answering 
tasks.', 'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}}

### Hybrid Search - Dense Index
For the Hybrid Search, we also need the dense index using the vector database, as we used in the previous steps.

#### Creaing the Dense Index

In [13]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer

qdrant_client = QdrantClient(
    ":memory:"
) 

# Create the embedding encoder
dense_encoder = SentenceTransformer('all-MiniLM-L6-v2') # Model to create embeddings

In [14]:
collection_name = "hybrid_search"

dense_index = qdrant_client.recreate_collection(
    collection_name=collection_name,
        vectors_config=models.VectorParams(
        size=dense_encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)
print(dense_index)

True


In [15]:
# vectorize!
qdrant_client.upload_points(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=dense_encoder.encode(doc["text"]).tolist(),
            payload=doc
        ) for idx, doc in enumerate(corpus_json) # data is the variable holding all the enriched texts
    ]
)

#### Searching the Dense Index
We will start with encoding the query with the dense encoder:

In [16]:
query_vector = dense_encoder.encode(query).tolist()

And use the encoded query to search the dense index:

In [18]:
dense_results = qdrant_client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=10
).points
console.print(dense_results)

[
    ScoredPoint(
        id=19,
        version=0,
        score=0.5686391495607938,
        payload={
            'id': 19,
            'text': '3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length 
Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 
24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance of Mixtral. (Left) Mixtral
has 100% retrieval accuracy of the Passkey task regardless of the location of the passkey and length of the input 
sequence. (Right) The perplexity of Mixtral on the proof-pile dataset decreases monotonically as the context length
increases.\n\nThis chunk is part of the "Results" section of the document, specifically discussing the long-range 
performance of the Mixtral 8x7B model on a passkey retrieval task and its perplexity on the proof-pile dataset, 
highlighting its effectiveness in handling long contexts.',
            'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
        },
        vector=None,
        shard_key=None,
        order_value=None
    ),
    ScoredPoint(
        id=48,
        version=0,
        score=0.48376391818471365,
        payload={
            'id': 48,
            'text': '13\n\nThe chunk appears to be part of the detailed results section of the document, 
specifically discussing the performance metrics and comparisons of the Mixtral 8x7B model against other models like
Llama 2 and GPT-3.5 across various benchmarks, highlighting its efficiency and effectiveness in different tasks.',
            'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
        },
        vector=None,
        shard_key=None,
        order_value=None
    ),
    ScoredPoint(
        id=4,
        version=0,
        score=0.4728943242453446,
        payload={
            'id': 4,
            'text': 'Instruct under the Apache 2.0 license1, free for academic and commercial usage, ensuring broad
accessibility and potential for diverse applications. To enable the community to run Mixtral with a fully 
open-source stack, we submitted changes to the vLLM project, which integrates Megablocks CUDA kernels for efficient
inference. Skypilot also allows the deployment of vLLM endpoints on any instance in the cloud. # 2 Architectural 
details Mixtral is based on a transformer architecture [31] and uses the same modifications as described in [18], 
with the notable exceptions that Mix- tral supports a fully dense context length of 32k tokens, and the feed- 
forward blocks are replaced by Mixture-of-Expert layers (Section 2.1). The model architecture parameters are 
summarized in Table 1.\n\nThis chunk is part of the "Introduction" and "Architectural details" sections of the 
document, which discusses the Mixtral 8x7B model, its licensing, accessibility, and the architectural modifications
made to the transformer framework, including the implementation of Mixture-of-Expert layers and context length 
specifications.',
            'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
        },
        vector=None,
        shard_key=None,
        order_value=None
    ),
    ScoredPoint(
        id=10,
        version=0,
        score=0.45750413551906866,
        payload={
            'id': 10,
            'text': '¢ Math: GSM8K [9] (8-shot) with maj@8 and MATH [17] (4-shot) with maj@4 â ¢ Code: Humaneval 
[4] (0-shot) and MBPP [1] (3-shot) â ¢ Popular aggregated results: MMLU [16] (5-shot), BBH [29] (3-shot), and AGI 
Eval [34] (3-5-shot, English multiple-choice questions only)\n\nThis chunk is part of the "Results" section of the 
document, where the authors compare the performance of the Mixtral model against other models across various 
benchmarks, specifically highlighting its capabilities in mathematics and code generation tasks.',

### Hybrid Search - Merging Results
There are a few options to merge the results from the two methods (sparse and dense). In this notebook, we will use a simple weighted average.

In [19]:
documents_with_scores = []
for hit in dense_results:
    doc_id = hit.payload["id"]
    doc_text = next((doc for doc in corpus_json if doc["id"] == doc_id), None)["text"]
    doc_dense_score = hit.score
    documents_with_scores.append({
        "id": doc_id,
        "text": doc_text,
        "dense_score": doc_dense_score
    })

for i, result in enumerate(sparse_results[0]):
    doc_id = result["id"]
    doc_text = next((doc for doc in corpus_json if doc["id"] == doc_id), None)["text"]
    doc_sparse_score = sparse_scores[0][i]
    for doc in documents_with_scores:
        if doc["id"] == doc_id:
            doc["sparse_score"] = doc_sparse_score
            break
console.print(documents_with_scores)


[
    {
        'id': 19,
        'text': '3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey 
Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 24K 28K 0 
5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance of Mixtral. (Left) Mixtral has 100% 
retrieval accuracy of the Passkey task regardless of the location of the passkey and length of the input sequence. 
(Right) The perplexity of Mixtral on the proof-pile dataset decreases monotonically as the context length 
increases.\n\nThis chunk is part of the "Results" section of the document, specifically discussing the long-range 
performance of the Mixtral 8x7B model on a passkey retrieval task and its perplexity on the proof-pile dataset, 
highlighting its effectiveness in handling long contexts.',
        'dense_score': 0.5686391495607938,
        'sparse_score': 1.0247643
    },
    {
        'id': 48,
        'text': '13\n\nThe chunk appears to be part of the detailed results section of the document, specifically 
discussing the performance metrics and comparisons of the Mixtral 8x7B model against other models like Llama 2 and 
GPT-3.5 across various benchmarks, highlighting its efficiency and effectiveness in different tasks.',
        'dense_score': 0.48376391818471365
    },
    {
        'id': 4,
        'text': 'Instruct under the Apache 2.0 license1, free for academic and commercial usage, ensuring broad 
accessibility and potential for diverse applications. To enable the community to run Mixtral with a fully 
open-source stack, we submitted changes to the vLLM project, which integrates Megablocks CUDA kernels for efficient
inference. Skypilot also allows the deployment of vLLM endpoints on any instance in the cloud. # 2 Architectural 
details Mixtral is based on a transformer architecture [31] and uses the same modifications as described in [18], 
with the notable exceptions that Mix- tral supports a fully dense context length of 32k tokens, and the feed- 
forward blocks are replaced by Mixture-of-Expert layers (Section 2.1). The model architecture parameters are 
summarized in Table 1.\n\nThis chunk is part of the "Introduction" and "Architectural details" sections of the 
document, which discusses the Mixtral 8x7B model, its licensing, accessibility, and the architectural modifications
made to the transformer framework, including the implementation of Mixture-of-Expert layers and context length 
specifications.',
        'dense_score': 0.4728943242453446,
        'sparse_score': 0.84697586
    },
    {
        'id': 10,
        'text': '¢ Math: GSM8K [9] (8-shot) with maj@8 and MATH [17] (4-shot) with maj@4 â ¢ Code: Humaneval [4] 
(0-shot) and MBPP [1] (3-shot) â ¢ Popular aggregated results: MMLU [16] (5-shot), BBH [29] (3-shot), and AGI Eval 
[34] (3-5-shot, English multiple-choice questions only)\n\nThis chunk is part of the "Results" section of the 
document, where the authors compare the performance of the Mixtral model against other models across various 
benchmarks, specifically highlighting its capabilities in mathematics and code generation tasks.',
        'dense_score': 0.45750413551906866
    },
    {
        'id': 2,
        'text': 'expertsâ ) to process the token and combine their output additively. This technique increases the 
number of parameters of a model while controlling cost and latency, as the model only uses a fraction of the total 
set of parameters per token. Mixtral is pretrained with multilingual data using a context size of 32k tokens. It 
either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several benchmarks. In particular, 
Mixture of Experts Layer i gating inputs af outputs router expert\n\nThis chunk is part of the section discussing 
the architecture and functionality of the Mixtral 8x7B model, specifically focusing on the Mixture of Experts (MoE)
mechanism, which allows the model to

We will normalize the scores of each index, and than calculate a weighted score that gives more weight (0.8) to the dense index.

In [20]:
import numpy as np

# Normalize the two types of scores
dense_scores = np.array([doc.get("dense_score", 0) for doc in documents_with_scores])
sparse_scores = np.array([doc.get("sparse_score", 0) for doc in documents_with_scores])

dense_scores_normalized = (dense_scores - np.min(dense_scores)) / (np.max(dense_scores) - np.min(dense_scores))
sparse_scores_normalized = (sparse_scores - np.min(sparse_scores)) / (np.max(sparse_scores) - np.min(sparse_scores))

# Calculate a weighted score with alpha of 0.2 to the sparse score
alpha = 0.2
weighted_scores = (1 - alpha) * dense_scores_normalized + alpha * sparse_scores_normalized

# Pick up the top 3 documents with the weighted score
top_docs = sorted(
    zip(
        documents_with_scores, 
        weighted_scores
    ), 
    key=lambda x: x[1], 
    reverse=True
)[:3]
console.print(top_docs)

[
    (
        {
            'id': 19,
            'text': '3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length 
Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 
24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance of Mixtral. (Left) Mixtral
has 100% retrieval accuracy of the Passkey task regardless of the location of the passkey and length of the input 
sequence. (Right) The perplexity of Mixtral on the proof-pile dataset decreases monotonically as the context length
increases.\n\nThis chunk is part of the "Results" section of the document, specifically discussing the long-range 
performance of the Mixtral 8x7B model on a passkey retrieval task and its perplexity on the proof-pile dataset, 
highlighting its effectiveness in handling long contexts.',
            'dense_score': 0.5686391495607938,
            'sparse_score': 1.0247643
        },
        0.9077696455115611
    ),
    (
        {
            'id': 4,
            'text': 'Instruct under the Apache 2.0 license1, free for academic and commercial usage, ensuring broad
accessibility and potential for diverse applications. To enable the community to run Mixtral with a fully 
open-source stack, we submitted changes to the vLLM project, which integrates Megablocks CUDA kernels for efficient
inference. Skypilot also allows the deployment of vLLM endpoints on any instance in the cloud. # 2 Architectural 
details Mixtral is based on a transformer architecture [31] and uses the same modifications as described in [18], 
with the notable exceptions that Mix- tral supports a fully dense context length of 32k tokens, and the feed- 
forward blocks are replaced by Mixture-of-Expert layers (Section 2.1). The model architecture parameters are 
summarized in Table 1.\n\nThis chunk is part of the "Introduction" and "Architectural details" sections of the 
document, which discusses the Mixtral 8x7B model, its licensing, accessibility, and the architectural modifications
made to the transformer framework, including the implementation of Mixture-of-Expert layers and context length 
specifications.',
            'dense_score': 0.4728943242453446,
            'sparse_score': 0.84697586
        },
        0.4584658824855766
    ),
    (
        {
            'id': 2,
            'text': 'expertsâ ) to process the token and combine their output additively. This technique increases 
the number of parameters of a model while controlling cost and latency, as the model only uses a fraction of the 
total set of parameters per token. Mixtral is pretrained with multilingual data using a context size of 32k tokens.
It either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several benchmarks. In particular, 
Mixture of Experts Layer i gating inputs af outputs router expert\n\nThis chunk is part of the section discussing 
the architecture and functionality of the Mixtral 8x7B model, specifically focusing on the Mixture of Experts (MoE)
mechanism, which allows the model to utilize a subset of its parameters for each token processed, enhancing 
efficiency and performance across various benchmarks.',
            'dense_score': 0.4468149922591207,
            'sparse_score': 1.7322638
        },
        0.43427722898555277
    )
]

### Using merged results to generate a reply
We can now take the merged results and call the LLM to generate the reply to the user's query.

In [21]:
# define a variable to hold the search results for the generation model
search_results = [doc[0]['text'] for doc in top_docs]

In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
# Now time to connect to the large language model
from openai import OpenAI
from rich.text import Text

client = OpenAI()
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are chatbot, an research expert. Your top priority is to help guide users to understand reserach papers."},
        {"role": "user", "content": query},
        {"role": "assistant", "content": str(search_results)}
    ]
)

response_text = Text(completion.choices[0].message.content)

In [24]:
from rich.panel import Panel

panel = Panel(response_text, title=f"Hybrid Search Reply to \"{query}\"")
console.print(panel)

╭─────────────────────────── Hybrid Search Reply to "What is context size of Mixtral?" ───────────────────────────╮
│ The context size of Mixtral is 32k tokens, which means it can effectively process a context of up to 32,000     │
│ tokens during its pretraining with multilingual data.                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Saving the retrieved documents to be used in the next reranking notebook, which demonstrates a more advanced method to merge Hybrid Search results.

In [25]:
import json

with open('data/dense_results.json', 'w') as f:
    json.dump([dense_result.payload for dense_result in dense_results], f, default=str)

with open('data/sparse_results.json', 'w') as f:
    json.dump([sparse_result for sparse_result in sparse_results[0]], f, default=str)